다음 셀은 결과 재현성을 보장하기 위해서 시드를 고정한다. 여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.

In [1]:
# 참고 - 시드 고정
import random
import numpy as np
import torch
 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 1-1 텐서 자료형

본 노트북은 본문 1-1절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.

- 텐서를 만드는 방법
- 차원과 형태 확인
- 요소 자료형 변환
- 텐서 연산과 브로드캐스팅
- 인덱싱과 슬라이싱
- 텐서 형태 변환과 차원 조작

## 리스트나 넘파이 배열로 텐서 만들기

`torch.tensor()` 함수에 리스트나 넘파이 배열을 인자로 전달해 텐서를 만들 수 있다.

In [2]:
########################################################################################
# 코드 1-1 - 리스트와 넘파이 배열로 텐서 만들기
########################################################################################
import torch
import numpy as np

data_list = [1., 1., 0., 0., 1., 1.]
data_nparray = np.array([[1, 1, 0], [0, 1, 1]])
data_3d = [[[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]],
           [[12, 13, 14, 15], [16, 17, 18, 19], [20, 21, 22, 23]]]

tensor_1d = torch.tensor(data_list)                 # 1차원 리스트로 1차원 텐서 생성
tensor_2d = torch.tensor(data_nparray)              # 2차원 넘파이 배열로 2차원 텐서 생성
tensor_3d = torch.tensor(data_3d)                   # 3차원 리스트로 3차원 텐서 생성

print(tensor_1d)
print(tensor_2d)
print(tensor_3d)

tensor([1., 1., 0., 0., 1., 1.])
tensor([[1, 1, 0],
        [0, 1, 1]])
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


- 인자로 전달한 리스트나 넘파이 배열과 같은 차원의 텐서가 생성된다.
- 텐서 요소의 자료형은 원본 자료형을 기반으로 자동으로 정해진다.
  - 단, 파이토치 고유의 자료형(`torch.float32`, `torch.int64` 등)으로 변환된다.
- 숫자 이외의 값(문자열 등)으로는 텐서를 만들 수 없다.

In [3]:
########################################################################################
# 코드 1-2 - 텐서 요소의 자료형 확인과 자료형 결정 규칙
########################################################################################

# dtype을 지정하지 않으면 자료형은 원래 자료형을 보존하는 방향으로 자동 결정됨
print(tensor_1d.dtype)                      # 실수 리스트는 float32형 텐서로 자동 결정
print(tensor_2d.dtype)                      # 넘파이 배열의 int64를 그대로 유지하며 변환

# 자료형 지정: dtype 인자로 직접 지정(별칭 사용 가능)
long_tensor = torch.tensor(data_list, dtype=torch.int64)
double_tensor = torch.tensor(data_nparray, dtype=torch.float64)
print(long_tensor.dtype)            
print(double_tensor.dtype)          

# 실수 리스트로 정수형 텐서를 만들면 부호와 상관없이 소수점 이하를 절삭
tensor_int64 = torch.tensor([3.14, 3.64, -3.14, -3.64], dtype=torch.int64)
print(tensor_int64)                 

# 정수와 실수가 섞인 리스트로 텐서를 만들면 형승격으로 실수형(float32) 텐서 생성
mixed_list = [1.0, 2]
mixed_tensor = torch.tensor(mixed_list)
print(mixed_tensor.dtype)  

# 숫자 이외의 값이 포함된 리스트로 텐서를 만들 수 없음
# torch.tensor(['Hello', 'TIE101'])         # ValueError 예외 발생


torch.float32
torch.int64
torch.int64
torch.float64
tensor([ 3,  3, -3, -3])
torch.float32


### 텐서 요소의 자료형 변경
- 텐서 생성 후에 `to()` 메서드 사용
  - `dtype` 인자에 지정하는 것이 원칙이지만
  - `to()` 메서드의 파라미터가 `dtype` 속성이므로 `dtype`을 생략해도 무방하다.
- 편의 메서드를 사용해도 된다.
  - `float()`: `torch.float32` 또는 `torch.float`로 변경
  - `double()`: `torch.float64` 또는 `torch.double`로 변경
  - `long()`: `torch.int64` 또는 `torch.long`으로 변경
- `to()` 메서드, 편의 메서드 모두 변경된 텐서를 **반환**하므로, 반드시 할당해 사용해야 한다.

In [4]:
########################################################################################
# 코드 1-3 - 텐서 요소의 자료형 변경
########################################################################################

# 형변환 후 할당해 사용해야 함
long_tensor = double_tensor.to(dtype=torch.long)  # int64형 텐서(long형 텐서)로 변경

# dtype 속성에 한해서 dtype 인자 키워드 생략 가능
long_tensor = double_tensor.to(torch.long)        # int64형 텐서로 변경

# int64, float32, float64는 형변환 메서드 long(), float(), double()로 변경 가능
float_tensor = double_tensor.float()
double_tensor = float_tensor.double()
long_tensor = double_tensor.long()

- 텐서 요소의 자료형은 메서드를 통해서만 변경 가능
  - `dtype` 속성에 할당 연산으로는 변경할 수 없음

In [5]:
# 참고 - 텐서의 dtype 속성은 메서드를 통해서만 변경 가능

# double_tensor.dtype = torch.long              # AttributeError 예외 발생

## 텐서의 차원과 형태

- 차원: `dim()` 메서드 또는 `ndim` 속성으로 확인
- 형태: `size()` 메서드 또는 `shape` 속성으로 확인
- 전체 요소의 수: `numel()` 메서드로 계산
- 스칼라 텐서: 숫자 하나로 만든 0차원 텐서
    - 스칼라 텐서의 형태: `torch.Size([])`
    - `torch.Size([1])` 또는 `(1,)` 형태의 텐서는 1차원 텐서

In [6]:
########################################################################################
# 코드 1-4 - 텐서의 차원, 형태, 요소 수 확인과 스칼라 텐서
########################################################################################

# 차원 확인: dim() 메서드 또는 ndim 속성
print(f'tensor_1d의 차원 : {tensor_1d.ndim}')          
print(f'tensor_3d의 차원 : {tensor_3d.dim()}')         

# 형태 확인: size() 메서드 또는 shape 속성
print(f'tensor_1d의 형태 : {tensor_1d.shape}')          
print(f'tensor_3d의 형태 : {tensor_3d.size()}')         

# 전체 요소 수: numel() 메서드(각 차원 크기의 곱)
print(f'tensor_3d의 전체 요소 수: {tensor_3d.numel()}') 

# 스칼라 텐서: 숫자 하나로 만든 0차원 텐서
scalar_tensor = torch.tensor(101)
print(f'스칼라 텐서 scalar_tensor: {scalar_tensor}')    
print(f'스칼라 텐서의 차원: {scalar_tensor.dim()}')     
print(f'스칼라 텐서의 형태: {scalar_tensor.size()}')    



tensor_1d의 차원 : 1
tensor_3d의 차원 : 3
tensor_1d의 형태 : torch.Size([6])
tensor_3d의 형태 : torch.Size([2, 3, 4])
tensor_3d의 전체 요소 수: 24
스칼라 텐서 scalar_tensor: 101
스칼라 텐서의 차원: 0
스칼라 텐서의 형태: torch.Size([])


- 각 차원의 크기가 일정하지 않으면 텐서로 변환할 수 없음

In [7]:
# 참고 - 텐서로 변환할 수 없는 리스트

irregular_list = [[[1], [2]], [[3], [4], [5]]]
# torch.tensor(irregular_list)  # ValueError 예외 발생 (첫 번째 차원 크기는 2, 두 번째 차원 크기는 2와 3으로 불규칙)

## 정해진 형태의 텐서 만들기

- 동일 값으로 채운 텐서를 만드는 함수
  - `torch.zeros(size)`: `0.`으로 채운 텐서 생성 
  - `torch.ones(size)`: `1.`으로 채운 텐서 생성
  - `size` 인자는 튜플, 리스트로 지정해도 되고, 언패킹해서 형태를 그대로 인자로 나열해도 됨
  - 생성된 텐서는 실수형 텐서로 정수형 텐서가 필요하면 `dtype` 인자를 지정하거나, `to()` 메서드를 사용해 변경해야 함


In [8]:
########################################################################################
# 코드 1-5 - 실수 0., 정수 0과 실수 1.로 채워진 텐서 만들기
########################################################################################

# 실수 0.으로 채워진 (2, 3) 형태의 텐서 생성
zeros_tensor = torch.zeros((2, 3))
# zeros(2, 3), zeros([2, 3])의 결과도 동일함

print(zeros_tensor)

# 정수 0으로 채워진 (2, 3) 형태의 torch.int64형 텐서 생성(dtype 인자로 자료형 지정)
long_zeros_tensor = torch.zeros(2, 3, dtype=torch.int64)
# torch.zeros(2, 3).to(torch.int64)와 동일한 결과
print(long_zeros_tensor)

# 실수 1.로 채워진 (5, ) 형태의 텐서 생성
ones_tensor = torch.ones((5, ))
print(ones_tensor)

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[0, 0, 0],
        [0, 0, 0]])
tensor([1., 1., 1., 1., 1.])


- 무작위 값으로 채워진 텐서를 만드는 함수
    - `torch.rand(size)`: 0과 1 사이의 균일 분포에서 무작위 값으로 채워진 `size` 형태의 텐서를 생성
    - `torch.randn(size)`: 표준정규분포를 따르는 무작위 값으로 채워진 `size` 형태의 텐서를 생성
    - `torch.randint(low, high, size)`: `low` 이상, `high` 미만의 정수형 난수로 구성된 `size` 형태의 텐서를 생성
        - 다른 생성 함수와 달리 `randint()` 함수의 `size` 인자는 반드시 튜플 또는 리스트로 지정해야 함
    - 무작위 값으로 채우더라도 `import random` 필요 없음(파이토치의 내장 난수 발생기 사용)

In [9]:
########################################################################################
# 코드 1-6 - 무작위 값으로 채워진 텐서 만들기
########################################################################################

# import random 필요 없음: torch 모듈만으로 난수 텐서 생성 가능
# (2, 2) 형태의 0~1 범위 실수형 난수 텐서 생성
print(torch.rand((2, 2)))       # torch.rand(2, 2)와 같음

# (2, 2) 형태의 표준정규분포를 따르는 실수형 난수 텐서 생성
print(torch.randn((2, 2)))      # torch.randn(2, 2)와 같음

# (2, 2) 형태의 1 이상 5 미만 정수형 난수 텐서 생성(상한은 포함되지 않음)
print(torch.randint(1, 5, (2, 2)))
# randint(1, 5, 2, 2)는 TypeError 예외 (인자 언패킹 미지원)

tensor([[0.8823, 0.9150],
        [0.3829, 0.9593]])
tensor([[ 0.2345,  0.2303],
        [-1.1229, -0.1863]])
tensor([[3, 3],
        [4, 1]])


- 일정한 간격의 값으로 구성된 텐서를 만드는 함수
    - `torch.arange(start, end, step)`: `start` 이상, `end` 미만 범위의 간격이 `step`인 등차수열로 구성된 1차원 텐서 생성
        - `step` 인자 생략시 1
        - `start`, `end`, `step`이 모두 정수인 경우 정수형 텐서 생성, 이외의 경우 실수형 텐서 생성
    - `torch.linspace(start, end, steps)`: `start`로 시작, `end`로 끝나고 요소의 수가 `steps`인 1차원 텐서 생성
        - `steps` 인자 생략할 수 없음
        - 항상 실수형 텐서 생성

In [10]:
########################################################################################
# 코드 1-7 - 일정한 간격의 값으로 구성된 텐서 만들기
########################################################################################

# 3씩 증가하는 등차수열 텐서 생성: 상한(10) 포함되지 않음
print(torch.arange(1, 10, 3))  

# 일정 간격으로 크기가 3인 텐서 생성: 상한(10) 포함
print(torch.linspace(1, 10, 3))

tensor([1, 4, 7])
tensor([ 1.0000,  5.5000, 10.0000])


### 추가 설명(`clamp()` 메서드)
- 지정된 범위 내의 요솟값을 가지는 텐서 생성
- 생성된 텐서의 범위를 제한할 때 유용

In [11]:
# 참고 - clamp() 메서드 사용 예시

# 설명에 적합한 난수가 발생되도록 시드 조정
torch.manual_seed(0)

t = torch.randn((3, ))                 
print(t) 

# 무작위 생성 텐서의 요소값 범위를 -1.0 ~ 1.0 범위로 제한
clamped = t.clamp(min=-1.0, max=1.0)     
# torch.clamp(t, min=-1.0, max=1.0) 와 같다
print(clamped) 

# min 또는 max만 지정 가능
clamped_min = t.clamp(min=0.0)
print(clamped_min)

tensor([ 1.5410, -0.2934, -2.1788])
tensor([ 1.0000, -0.2934, -1.0000])
tensor([1.5410, 0.0000, 0.0000])


## 텐서 연산과 브로드캐스팅

- 텐서 사이의 사칙 연산: 같은 위치 요소끼리 계산하는 요소별 연산
    - 연산 결과 텐서의 자료형은 파이썬 연산 결과의 자료형을 기준으로 결정됨
- 형태가 같은 텐서끼리만 연산 가능
- 단, 브로드캐스팅이 가능한 경우 형태가 다른 텐서의 연산도 가능
    - 브로드캐스팅으로 형태를 맞춘 후 요소별 연산


In [12]:
########################################################################################
# 코드 1-8 - 텐서끼리의 연산
########################################################################################

# 형태가 같은 두 텐서 사이의 덧셈과 나눗셈 연산
operand1 = torch.tensor([[8, 6], [4, 2]])
operand2 = torch.tensor([[4, 3], [2, 1]])

# 연산 결과 텐서의 자료형은 파이썬 연산 결과 자료형과 일치
print(operand1 + operand2)
print(operand1 / operand2)

tensor([[12,  9],
        [ 6,  3]])
tensor([[2., 2.],
        [2., 2.]])


In [13]:
########################################################################################
# 코드 1-9 - 텐서의 브로드캐스팅 연산
########################################################################################

# 텐서와 스칼라의 브로드캐스팅: 스칼라를 텐서와 같은 형태로 확장
print(operand1 - 5)             
print(torch.tensor(5) - operand1) 

# 형태가 다른 두 텐서 사이의 브로드캐스팅
# (2, 1, 2) 텐서와 (2, 2) 텐서 사이의 연산: 두 텐서를 (2, 2, 2)로 맞춘 후 요소별 연산
tensor_2x1x2 = torch.tensor([[[1, 2]], [[3, 4]]])  # (2, 1, 2)
tensor_2x2 = torch.tensor([[5, 6], [7, 8]])        # (2, 2)
print(tensor_2x1x2 + tensor_2x2)

tensor([[ 3,  1],
        [-1, -3]])
tensor([[-3, -1],
        [ 1,  3]])
tensor([[[ 6,  8],
         [ 8, 10]],

        [[ 8, 10],
         [10, 12]]])


- 브로드캐스팅이 불가능한 텐서끼리 연산: 예외 발생

In [14]:
# 참고 - 브로드캐스팅이 불가능한 텐서끼리의 연산
# 브로드캐스팅 2단계에서 크기 3과 2가 충돌
# torch.randn((1, 3, 2)) + torch.randn((2, 1))        # RuntimeError 예외 발생 (브로드캐스팅 불가능)


## 텐서 인덱싱과 슬라이싱

- 텐서의 인덱싱과 슬라이싱은 넘파이 배열의 방식을 따름
- `t[0][1]` 형식과 `t[0, 1]` 형식 모두 지원(후자의 넘파이 배열 스타일을 많이 사용함)
- 인덱싱의 결과는 항상 텐서(스칼라 텐서 또는 일반 텐서, 일반 정수 결과는 나오지 않음)
- 슬라이싱의 결과는 항상 일반 텐서


In [15]:
########################################################################################
# 코드 1-10 - 텐서의 인덱싱과 슬라이싱
########################################################################################

t1 = torch.tensor([0, 1, 2, 3, 4])
t2 = torch.tensor([[0, 1, 2], [3, 4, 5], [6, 7, 8]])

# 인덱싱
print(t1[0])                  # 첫 번째 요소, 스칼라 텐서
print(t2[-1])                 # 마지막 요소(음수 인덱싱), 일반 텐서
print(t2[0])                  # 이차원 텐서의 첫 번째 행
print(t2[0, 1])               # t2[0][1]과 동일

# 슬라이싱
print(t1[1:-1])               # 슬라이싱
print(t1[::2])                # 슬라이싱 (간격 지정)
print(t2[:, 1:])              # 모든 행의 1번 열 이후
print(t2[:, ::2])             # 모든 행의 짝수 열

tensor(0)
tensor([6, 7, 8])
tensor([0, 1, 2])
tensor(1)
tensor([1, 2, 3])
tensor([0, 2, 4])
tensor([[1, 2],
        [4, 5],
        [7, 8]])
tensor([[0, 2],
        [3, 5],
        [6, 8]])


- 인덱스로 리스트나 텐서를 사용할 수 있음
    - 여러 요소를 한 번에 선택
    - `t[1, 2]`와 `t[[1, 2]]`는 다르므로 주의

In [16]:
########################################################################################
# 코드 1-11 - 리스트나 텐서를 인덱스로 사용하는 인덱싱
########################################################################################

print(t1[[0, 2]])                    # 인덱스가 0과 2인 요소를 선택
print(t1[torch.tensor([0, 2])])      # tensor([0, 2]): 정수형 텐서도 사용 가능

# 다음 둘은 다르므로 주의해야 함
print(t2[[1, 2]])                    # tensor([[3, 4, 5], [6, 7, 8]]): 1번, 2번 행 선택
print(t2[1, 2])                      # tensor(5): (1, 2) 위치의 요소 하나를 인덱싱

print(t2[[0, 1], [1, 2]])            # tensor([1, 5]): (0, 1)과 (1, 2) 위치를 각각 선택

tensor([0, 2])
tensor([0, 2])
tensor([[3, 4, 5],
        [6, 7, 8]])
tensor(5)
tensor([1, 5])


- 불리언 마스크 인덱싱
  - 마스크를 사용해 조건을 만족하는 요소만 골라내는 인덱싱

In [17]:
########################################################################################
# 코드 1-12 - 마스크와 불리언 마스크 인덱싱
########################################################################################

# 불리언 마스크는 각 위치의 요소가 조건에 맞는지 여부로 구성된 참거짓형 텐서
# 2를 초과하는 요소의 위치만 True인 마스크 생성
mask = t1 > 2      # 마스크: 각 위치의 요소가 조건에 맞는지 여부로 구성된 참거짓형 텐서
print(mask)        
print(t1[mask])    # True에 대응되는 요소 인덱싱

# 같은 형태의 다른 텐서에 마스크를 적용해 인덱싱 가능
t3 = torch.tensor([0, 10, 20, 30, 40])
print(t3[mask])   

tensor([False, False, False,  True,  True])
tensor([3, 4])
tensor([30, 40])


- 인덱싱이나 슬라이싱에 할당 연산으로 텐서 요소의 값을 변경할 수 있음
- 슬라이싱은 참조로 동작하므로, 슬라이싱한 텐서의 요소를 변경하면 원본 텐서의 요소도 따라서 바뀜
    - 원본과 분리된 새로운 텐서는 `clone()` 메서드로 복사해 생성할 수 있음

In [18]:
########################################################################################
# 코드 1-13 - 인덱싱, 슬라이싱으로 요솟값 변경
########################################################################################

# 인덱싱과 할당 연산으로 요솟값 변경 
t1[0] = -1                                 # tensor([-1, 1, 2, 3, 4])으로 바뀜
t2[-1, -1] = -8                            # tensor([[...], [...], [6, 7, -8]])로 바뀜
print(t1)

# 슬라이싱은 참조로 동작: 슬라이싱한 텐서의 요소를 할당 연산으로 바꾸면 원본도 따라 바뀜
t4 = torch.tensor([0, 1, 2, 3, 4])
sliced_tensor = t4[:2]                     # tensor([0, 1]): 슬라이싱(복사가 아닌 참조)
sliced_tensor[0] = -1                      # t4[0] = -1과 같음
print(sliced_tensor)                       
print(t4)                                  

# clone()으로 복사하면 원본과 분리됨 
copied_tensor = t4[:2].clone()             # tensor([-1, 1]): 슬라이싱 후 복사
copied_tensor[0] = 0                       # t4의 요솟값은 바뀌지 않음
print(copied_tensor)                       
print(t4)                                  

tensor([-1,  1,  2,  3,  4])
tensor([-1,  1])
tensor([-1,  1,  2,  3,  4])
tensor([0, 1])
tensor([-1,  1,  2,  3,  4])


In [19]:
# 참고 - 슬라이싱과 할당 연산으로 요솟값 변경(각주 7)
# 할당 대상은 1개 또는 슬라이싱 영역의 요소와 같은 수로 구성되어야 하며
# 여러 개의 요소를 지정할 때는 텐서로 지정해야 함
t5 = torch.tensor([0, 1, 2, 3, 4])
t5[1:-1] = 11                           # 슬라이싱된 3개의 요소 모두를 11로 변경
print(t5)   

t5[1:-1] = torch.tensor([22, 33, 44])   # 슬라이싱된 3개의 요소 모두를 새로운 값으로 변경
print(t5)

# t5[1:-1] = [22, 33, 44]               # 슬라이싱에 리스트를 할당하면 예외 발생
# t5[1:-1] = torch.tensor([22, 33])     # 슬라이싱된 요소는 3개인데, 크기 2의 텐서를 할당해 예외 발생

tensor([ 0, 11, 11, 11,  4])
tensor([ 0, 22, 33, 44,  4])


## 텐서 형태의 변환

- `reshape(size)` 메서드: 요소들이 일렬로 쭉 늘어서 있다고 보고, 그 위에 순서대로 대괄호를 씌워 지정한 형태로 만들어 반환
    - `size` 인자는 `torch.rand()` 메서드의 `size` 인자와 사용법이 동일
    - 변형 전 후의 요소 수가 일치하도록 형태 인자를 지정해야 함

In [20]:
########################################################################################
# 코드 1-14 - 텐서의 형태를 바꾸는 reshape()
########################################################################################

flat_tensor = torch.arange(24)              # 0 ~ 23의 크기 24인 1차원 텐서
reshaped_3d = flat_tensor.reshape(2, 3, 4)  # (2, 3, 4) 형태의 3차원 텐서로 변형
reshaped_2d = reshaped_3d.reshape(2, 12)    # (2, 12) 형태의 2차원 텐서로 변형
                                            # flat_tensor.reshape(2, 12)와 동일
# reshaped_2d.reshape(7, 3): RuntimeError 예외, 요소의 수가 일치하도록 형태를 지정해야 함
print(reshaped_3d)  
print(reshaped_2d)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]])


- `flatten()` 메서드: 차원을 이어 붙이는 평탄화 메서드

In [21]:

########################################################################################
# 코드 1-15 - 텐서의 차원을 이어 붙이는 평탄화 flatten()
########################################################################################

reshaped_3d = torch.arange(24).reshape(2, 3, 4)
# 1차원 텐서로 평탄화 
flattened = reshaped_3d.flatten()               # tensor([0, 1, ..., 23])

# 첫 번째 차원에서 두 번째 차원까지만 이어 붙여 평탄화
front_flattened = reshaped_3d.flatten(0, 1)     # tensor([[0, 1, 2, 3], [4, 5, 6, 7], ...])
print(front_flattened.shape)                   

# 두 번째 차원에서 마지막(세 번째) 차원까지만 이어 붙여 평탄화
# -1: 뒤에서부터 첫 번째 차원(마지막 차원)을 의미
back_flattened = reshaped_3d.flatten(1, -1)     # tensor([[0, 1, ..., 11], [12, 13, ..., 23]])
print(back_flattened.shape)                    


torch.Size([6, 4])
torch.Size([2, 12])


## 차원의 순서 변경

- `transpose()` 메서드: 두 차원의 순서를 맞바꾸는 메서드(맞바꿀 두 차원의 인덱스를 인자로 지정)
- `permute()` 메서드: 모든 차원의 순서를 일괄 변경하는 메서드(새로운 차원 순서를 인자로 지정)
    - 변경하지 않는 차원도 같은 값으로 인자를 지정해야 함
- `t()` 메서드: 2차원 텐서의 두 차원 순서를 바꾼 전치 행렬 텐서 생성

In [22]:
########################################################################################
# 코드 1-16 - 텐서의 차원 순서 바꾸기
########################################################################################

base_2d = torch.arange(24).reshape(2, 12)
base_3d = torch.arange(24).reshape(2, 3, 4)

# transpose(): 두 차원의 순서 맞바꿈(맞바꿀 두 차원의 인덱스를 인자로 지정)
transposed = base_3d.transpose(0, 2)     # 첫 번째(0)와 세 번째(2) 차원 맞바꾸기
print(transposed.shape)                  
print(transposed[2, 1, 1])               # base_3d[1, 1, 2]와 같음

# permute(): 모든 차원의 순서를 일괄 변경(새로운 차원 순서를 인자로 지정)
permuted = base_3d.permute(1, 2, 0)      # 차원의 순서 (0, 1, 2)를 (1, 2, 0)으로 변경
print(permuted.shape)                    
# permute(1, 0)은 차원 수 불일치, permute(2, 3, 1)은 없는 차원(3)을 지정해 예외 발생

# t(): 2차원 텐서의 전치 행렬 텐서 생성
print(base_2d.shape, base_2d.t().shape)  


torch.Size([4, 3, 2])
tensor(18)
torch.Size([3, 4, 2])
torch.Size([2, 12]) torch.Size([12, 2])


## 차원의 추가와 삭제

- `unsqueeze()` 메서드: 크기 1인 차원을 추가
- `squeeze()` 크기 1인 차원을 제거
  - 제거할 차원의 크기가 1이 아닌 경우 제거하지 않음(예외 발생도 없음)

In [ ]:
########################################################################################
# 코드 1-17 - 텐서에 차원 추가하기
########################################################################################

base_2d = torch.arange(6).reshape(2, 3)    # (2, 3) 형태의 텐서
tensor_3d_1 = base_2d.unsqueeze(0)         # 첫 번째 차원을 추가
tensor_3d_2 = base_2d.unsqueeze(1)         # 두 번째 차원을 추가
tensor_3d_3 = base_2d.unsqueeze(-1)        # 마지막 차원(-1)을 추가                                           
# base_2d.unsqueeze(3): IndexError 예외(2차원 텐서에 네 번째 차원을 추가할 수 없음)

print(tensor_3d_1.shape, tensor_3d_2.shape, tensor_3d_3.shape)


torch.Size([1, 2, 3]) torch.Size([2, 1, 3]) torch.Size([2, 3, 1])


In [24]:
########################################################################################
# 코드 1-18 - 텐서의 차원 제거하기
########################################################################################

base_4d = torch.randn((1, 2, 1, 3))     # (1, 2, 1, 3) 형태의 4차원 텐서
dim_removed_1 = base_4d.squeeze()       # (2, 3) 형태의 텐서: 크기가 1인 모든 차원 제거
dim_removed_2 = base_4d.squeeze(2)      # (1, 2, 3) 형태의 텐서: 세 번째 차원(2) 제거
not_changed = base_4d.squeeze(-1)       # 제거된 차원 없음 (마지막 차원의 크기가 3)

print(dim_removed_1.shape, dim_removed_2.shape, not_changed.shape)

torch.Size([2, 3]) torch.Size([1, 2, 3]) torch.Size([1, 2, 1, 3])


## 여러 텐서를 이어 붙이기

- `torch.cat()`: 지정한 축을 따라 텐서를 이어 붙이는 함수
- `torch.stack()`: 새 차원을 추가해 텐서를 쌓는 함수

In [25]:
########################################################################################
# 코드 1-19 - 둘 이상의 1차원 텐서 이어 붙이기
########################################################################################

t1 = torch.tensor([1, 2, 3])
t2 = torch.tensor([4, 5, 6])
t3 = torch.tensor([7, 8])
# 이어 붙일 텐서를 리스트로 묶어서 인자로 전달
print(torch.cat([t1, t2]))                      # torch.tensor([1, 2, 3, 4, 5, 6])
# torch.cat(t1, t2)는 예외 발생

# 셋 이상의 텐서도 이어 붙일 수 있음
print(torch.cat([t1, t2, t3]))                  # torch.tensor([1, 2, 3, 4, 5, 6, 7, 8])



tensor([1, 2, 3, 4, 5, 6])
tensor([1, 2, 3, 4, 5, 6, 7, 8])


In [26]:
########################################################################################
# 코드 1-20 - 2차원 텐서 이어 붙이기
########################################################################################

t4 = torch.tensor([[1, 2], [3, 4]])
t5 = torch.tensor([[5, 6], [7, 8]])
cat_dim0 = torch.cat([t4, t5], dim=0)   # (4, 2) 형태의 텐서. dim=0은 생략 가능(기본값)
cat_dim1 = torch.cat([t4, t5], dim=1)   # (2, 4) 형태의 텐서

print(cat_dim0)
print(cat_dim1)

tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])
tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]])


In [27]:
########################################################################################
# 코드 1-21 - 2차원 텐서 쌓기
########################################################################################

stack_dim0 = torch.stack([t4, t5])            # (2, 2, 2), dim=0 생략
stack_dim1 = torch.stack([t4, t5], dim=1)     # 형태는 같지만 축에 따라 배치가 달라짐

print(stack_dim0)
print(stack_dim1)

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
tensor([[[1, 2],
         [5, 6]],

        [[3, 4],
         [7, 8]]])


## 연습 문제 

In [28]:
########################################################################################
# 코드 1-22 - MNIST 데이터셋 불러오기 (연습 문제 1-1)
########################################################################################

import torch
from torchvision import datasets, transforms

DATA_ROOT = '../../download'
transform = transforms.ToTensor()
train_set = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset=train_set, batch_size=32, shuffle=True)
images, labels = next(iter(train_loader))

# images와 labels 두 텐서의 형태를 출력해 보자.
